# Running the Rubin DP2 SkyKatana Footprint, Dust & Star Mask Pipeline Locally via IVOA TAP

**Environment**: Local `lsdb_env-py313` (no RSP container or EUPS `lsst_distrib` required)  
**Services**:
- **Rubin RSP External TAP** (`https://data.lsst.cloud/api/tap`, authenticated via an RSP Bearer token with `read:tap` scope)
- **ESA Gaia DR3 Public TAP** (`https://gea.esac.esa.int/tap-server/tap`, zero credentials required)
- **Automatic Offline Fallback**: `SyntheticDP2TapService` when `~/.rsp-tap.token` / `RSP_ACCESS_TOKEN` is not yet configured

---

## 1. Can Notebook 01 Run Locally Using TAP Services? (Architecture & Exact Mapping)

**Yes — with an important distinction between Butler binary maps and relational TAP tables:**

Inside the Rubin Science Platform (Notebook 01), Gen3 Butler reads binary `HealSparseMap` survey property maps (`sp_maps`) and `htm7`-sharded `the_monster` `SimpleCatalog` files directly from object storage. Those binary files are **not** stored as SQL rows in TAP. However, Rubin exposes all calibrated detector metadata, `ObsCore` coadd footprints, and `dp2.Object` catalogs (including SFD98 `ebv`) externally via **`https://data.lsst.cloud/api/tap`**, allowing every `SkyMaskPipe` stage to be built locally on your laptop:

| `SkyMaskPipe` Stage | Inside RSP (`01_DP2_Skykatana_Mask_RSP.ipynb`) | Local Execution via TAP (`02_DP2_Skykatana_Mask_Local_TAP.ipynb`) |
| :--- | :--- | :--- |
| **Authentication** | Automatic Nublado pod credentials | `~/.rsp-tap.token` or `$RSP_ACCESS_TOKEN` (`read:tap` scope) via `pyvo` or built-in `requests + VOTable` |
| **Stage 1: `propmask`** | Butler `deepCoadd_*_consolidated_map_*` | Stacks `dp2.VisitDetector` 4-corner CCD polygons (`band`, `expTime`, `seeing`, $5\sigma$ `magLim`) + `ivoa.ObsCore` `lsst.deep_coadd` polygons (`calib_level = 3`) onto HEALPix with inverse-variance coadd depth $m_{b,\mathrm{coadd}}^{\mathrm{obs}} = 1.25\log_{10}(\sum_i 10^{0.8\,m_{i,\mathrm{lim}}})$ and **Tutorial 311.1** dust de-reddening |
| **Stage 2: `dustmask`** | `rubin_scheduler` SFD98 / Butler map | Queries `dp2.Object.ebv` (SFD98) via RSP TAP (or local `dustmaps` / HEALPix SFD98 map) to mask $E(B-V)$ > `max_ebv` and $|b|$ < `min_galactic_lat_deg` |
| **Stage 3: `starmask` + `boxmask`** | Butler `the_monster_20250219` shards | Queries public **ESA Gaia DR3 TAP** (`phot_g_mean_mag`, `bp_rp`), transforms $(G, G_{\mathrm{BP}}-G_{\mathrm{RP}}) \to$ synthetic LSST `ugrizy` AB magnitudes, and runs `skykatana_lsst_radfunction` |
| **Stage 4: `combine` & Science** | `SkyMaskPipe.combine`, `makerans`, `apply` | Identical bit-packed `SkyMaskPipe.combine`, multi-scale visualizations, `makerans`, and `apply` filtering on `dp2.Object` galaxies queried via TAP |

### One-Time RSP Token Setup for Live DP2 TAP Queries
1. Log in to https://data.lsst.cloud → **Account** (top right) → **Security tokens** → **Create Token**.
2. Enable scope **`read:tap`** and save the token locally (never paste tokens into Git-tracked notebooks):
   ```bash
   printf '%s\n' 'gt-YOUR_RSP_TOKEN_HERE' > ~/.rsp-tap.token
   chmod 600 ~/.rsp-tap.token
   ```
*(Optional: install `pyvo` in `lsdb_env-py313` via `pip install pyvo`; if `pyvo` is not installed, `dp2_skykatana_mask` automatically uses its built-in IVOA TAP 1.1 `requests + Astropy VOTable` client!)*

In [ ]:
import json
import logging
from pathlib import Path
import sys

import astropy.units as u
from astropy.coordinates import Angle, SkyCoord
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'configs').exists() and (REPO_ROOT.parent / 'configs').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from skykatana import SkyMaskPipe
from dp2_skykatana_mask import (
    DEFAULT_GAIA_TAP_URL,
    DEFAULT_RSP_TAP_URL,
    MemoryConfig,
    build_dust_extinction_mask,
    build_local_tap_star_mask,
    build_tap_ebv_provider,
    build_tap_property_mask,
    compute_sky_extent,
    deredden_magnitude,
    discover_dp2_tap_schema,
    gaia_to_synthetic_lsst_magnitudes,
    get_rubin_band_extinction_rx,
    load_config,
    open_rsp_tap_client,
    plot_complete_mask_allsky,
    plot_complete_mask_fracmap,
    plot_complete_mask_multiscale,
    skykatana_lsst_radfunction,
)
from dp2_skykatana_mask.memory import MemoryGuard

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)-7s | %(message)s', datefmt='%H:%M:%S')
OUT_DIR = REPO_ROOT / 'outputs' / 'local_tap_ecdfs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Repository root :', REPO_ROOT)
print('Local output dir:', OUT_DIR)

## 2. Connect to RSP External TAP (`https://data.lsst.cloud/api/tap`) & Discover DP2 Schema

`open_rsp_tap_client()` checks `~/.rsp-tap.token` and `$RSP_ACCESS_TOKEN`:
- **Live RSP Mode**: Authenticates against `https://data.lsst.cloud/api/tap` with `Authorization: Bearer <token>` and inspects `TAP_SCHEMA.tables` / `TAP_SCHEMA.columns` to resolve `dp2.VisitDetector`, `dp2.Visit`, `dp2.Object`, and `ivoa.ObsCore` automatically.
- **Offline Demo Mode**: If no token is found on disk yet, seamlessly initializes `SyntheticDP2TapService` (containing dithered 6-band ECDFS `VisitDetector` exposures, `ivoa.ObsCore` `lsst.deep_coadd` patch polygons, 18,000 reddened `dp2.Object` galaxies with a NE Galactic cirrus filament, and Gaia DR3 bright stars) so you can run the entire notebook offline immediately.

In [ ]:
local_cfg = load_config(REPO_ROOT / 'configs' / 'dp2_ecdfs_pilot.yaml')
local_cfg.output.output_dir = str(OUT_DIR)
# Order 13 (nside=8192, ~25.8" pixels) is crisp & sub-second locally; raise to 14 or 15 for science production!
local_cfg.property_maps.order_sparse = 13
local_cfg.dust.order_sparse = 13
local_cfg.monster_stars.order_sparse = 13
local_cfg.output.order_out = 13

tap_client = open_rsp_tap_client(
    tap_url=DEFAULT_RSP_TAP_URL,
    token_file=local_cfg.access_token_file,  # None checks ~/.rsp-tap.token & $RSP_ACCESS_TOKEN
    allow_synthetic_fallback=True,
    synthetic_kwargs={
        'center_ra_deg': local_cfg.region.ra_deg,
        'center_dec_deg': local_cfg.region.dec_deg,
        'radius_deg': local_cfg.region.radius_deg,
    },
)

schema_info = discover_dp2_tap_schema(tap_client)
print(f'TAP client endpoint : {tap_client.tap_url} (backend={tap_client.backend_name}, synthetic={tap_client.is_synthetic})')
print(f'Discovered schema   : {schema_info}')

## 3. Reconstruct Dust-Corrected Coadd Property Maps (`propmask`) & `dustmask` via TAP

### How Coadd Depth & Exposure Maps Are Built from `dp2.VisitDetector` + `ivoa.ObsCore`
Each row $i$ in `dp2.VisitDetector` provides a calibrated single-CCD exposure's 4 sky corners `(llcra, llcdec, ulcra, ulcdec, urcra, urcdec, lrcra, lrcdec)`, filter `band`, `expTime`, PSF `seeing` (FWHM in arcseconds), and single-visit $5\sigma$ point-source depth $m_{i,\mathrm{lim}}$.

For every HEALPix pixel $(\alpha, \delta)$ at `order_sparse` covered by $N_b(\alpha, \delta)$ overlapping detector exposures in band $b$, background-dominated inverse-variance coaddition ($\sigma_{f,i}^{-2} \propto 10^{0.8\,m_{i,\mathrm{lim}}}$) yields:

$$m_{b,\mathrm{coadd}}^{\mathrm{obs}}(\alpha, \delta) = 1.25 \log_{10}\!\left(\sum_{i=1}^{N_b(\alpha,\delta)} 10^{0.8\, m_{i,\mathrm{lim}}}\right), \qquad m_{b,\mathrm{coadd}}^{\mathrm{dered}}(\alpha, \delta) = m_{b,\mathrm{coadd}}^{\mathrm{obs}}(\alpha, \delta) - R_b \cdot E(B-V)(\alpha, \delta).$$

*(Sanity check: stacking $N$ identical visits of depth $m_0$ gives $m_0 + 1.25\log_{10}(N) = m_0 + 2.5\log_{10}(\sqrt{N})$, matching exact $\mathrm{SNR} \propto \sqrt{N}$ scaling while respecting per-detector sky noise, zero-point variations, `ivoa.ObsCore` `lsst.deep_coadd` patch boundaries, and **DP2 Tutorial 311.1** foreground Galactic extinction!)*

In [ ]:
guard = MemoryGuard(
    max_ram_gib=local_cfg.memory.max_ram_gib,
    soft_warning_fraction=local_cfg.memory.soft_warning_fraction,
)
mkp_local = SkyMaskPipe(
    order_out=local_cfg.output.order_out,
    order_cov=local_cfg.output.order_cov,
)

# 1. Build SFD98 E(B-V) provider from local dustmaps/SFD or RSP TAP `dp2.Object.ebv`
rx = get_rubin_band_extinction_rx(r_v=local_cfg.dust.r_v)
ebv_provider = build_tap_ebv_provider(tap_client, local_cfg, schema_info=schema_info)
print('Active E(B-V) source:', ebv_provider.source_name)
print('Rubin R_b = A_b / E(B-V):', {b: round(v, 3) for b, v in rx.items()})

# 2. Stack `dp2.VisitDetector` polygons (+ `ivoa.ObsCore` deep_coadd patches) -> `propmask`
propmask, prop_stats, coadd_maps = build_tap_property_mask(
    pipe=mkp_local,
    tap_client=tap_client,
    cfg=local_cfg,
    guard=guard,
    ebv_provider=ebv_provider,
    schema_info=schema_info,
    output_stage='propmask',
)

# 3. Build negative `dustmask` stage excluding high-E(B-V) / low-|b| sightlines
dustmask, dust_stats = build_dust_extinction_mask(
    pipe=mkp_local,
    footprint_mask=propmask,
    cfg=local_cfg,
    ebv_provider=ebv_provider,
    guard=guard,
    rx_map=rx,
    output_stage='dustmask',
)

print(f'\nTAP propmask valid area : {prop_stats.area_deg2:.4f} deg² | per-band: {prop_stats.per_band_area_deg2}')
print(f'TAP dustmask vetoed area: {dust_stats.dustmask_area_deg2:.4f} deg² (max E(B-V) in footprint = {dust_stats.max_ebv_in_footprint})')

In [ ]:
# Visualize TAP-stacked observed vs. dust-dereddened r-band 5-sigma coadd depth!
if 'maglim_r_obs' in coadd_maps and 'maglim_r_dered' in coadd_maps:
    m_obs = coadd_maps['maglim_r_obs']
    m_der = coadd_maps['maglim_r_dered']
    vpix = m_obs.valid_pixels
    import hpgeom as hpg
    p_ra, p_dec = hpg.pixel_to_angle(m_obs.nside_sparse, vpix, nest=True, degrees=True)
    v_obs = m_obs.get_values_pix(vpix)
    v_der = m_der.get_values_pix(vpix)
    vmin_r = float(np.nanpercentile(v_der, 3))
    vmax_r = float(np.nanpercentile(v_obs, 97))

    fig, axes = plt.subplots(1, 3, figsize=(15.2, 4.5), constrained_layout=True)
    sc0 = axes[0].scatter(p_ra, p_dec, c=v_obs, s=1.0, cmap='viridis', vmin=vmin_r, vmax=vmax_r, rasterized=True)
    axes[0].invert_xaxis()
    axes[0].set_title('TAP-Stacked Observed r-band m_lim [AB]')
    axes[0].set_xlabel('RA [deg]'); axes[0].set_ylabel('Dec [deg]')
    fig.colorbar(sc0, ax=axes[0], label='mag')

    sc1 = axes[1].scatter(p_ra, p_dec, c=v_obs - v_der, s=1.0, cmap='magma', rasterized=True)
    axes[1].invert_xaxis()
    axes[1].set_title('Foreground Extinction A_r = R_r E(B-V)')
    axes[1].set_xlabel('RA [deg]')
    fig.colorbar(sc1, ax=axes[1], label='A_r [mag]')

    sc2 = axes[2].scatter(p_ra, p_dec, c=v_der, s=1.0, cmap='viridis', vmin=vmin_r, vmax=vmax_r, rasterized=True)
    axes[2].invert_xaxis()
    axes[2].set_title('Dust-Dereddened Intrinsic r-band m_lim [AB]')
    axes[2].set_xlabel('RA [deg]')
    fig.colorbar(sc2, ax=axes[2], label='mag')
    plt.show()

## 4. Bright-Star Mask (`starmask` + `boxmask`) Locally via Gaia DR3 TAP + `skykatana_lsst_radfunction`

On the RSP, `the_monster` (DMTN-277) combines Gaia DR3 positions with synthetic LSST fluxes derived from Gaia photometry/XP spectra. When running locally without Butler access to `the_monster` shards, `build_local_tap_star_mask`:
1. Queries public **ESA Gaia DR3 TAP** (`https://gea.esac.esa.int/tap-server/tap`, or `SyntheticDP2TapService` when running offline) for stars across the buffered footprint (`ra, dec, phot_g_mean_mag, bp_rp`).
2. Transforms $(G, G_{\mathrm{BP}} - G_{\mathrm{RP}}) \to$ synthetic apparent LSST `ugrizy` AB magnitudes (`gaia_to_synthetic_lsst_magnitudes`).
3. Applies our exact **`skykatana_lsst_radfunction`** (`R(m_LSST)` in degrees) and pixelizes circular bright-star halos (`starmask`) and saturated CCD bleed-trail boxes (`boxmask`) into `mkp_local`.

*(Note: when running live locally with internet access, `build_local_tap_star_mask` queries live ESA Gaia DR3 TAP automatically if `tap_client.is_synthetic` is `False`, or you can pass `gaia_tap_client=IVOATapClient(DEFAULT_GAIA_TAP_URL)` explicitly even in hybrid demos!)*

In [ ]:
star_info = build_local_tap_star_mask(
    pipe=mkp_local,
    footprint_mask=mkp_local.propmask,
    cfg=local_cfg,
    guard=guard,
    work_dir=OUT_DIR,
    rsp_tap_client=tap_client,
    star_stage='starmask',
    box_stage='boxmask',
)
print('Bright-star TAP summary:', json.dumps(star_info, indent=2))

# Combine positive `propmask` minus negative (`dustmask`, `starmask`, `boxmask`)
neg_stages = ['starmask']
if getattr(mkp_local, 'dustmask', None) is not None and int(mkp_local.dustmask.n_valid) > 0:
    neg_stages.insert(0, 'dustmask')
if getattr(mkp_local, 'boxmask', None) is not None and int(mkp_local.boxmask.n_valid) > 0:
    neg_stages.append('boxmask')

mkp_local.combine(
    positive=['propmask'],
    negative=neg_stages,
    order_out=local_cfg.output.order_out,
    order_cov=local_cfg.output.order_cov,
    bit_packed=True,
    verbose=True,
)

# Persist the SkyKatana pipeline + IVOA MOC locally
pipe_dir = OUT_DIR / 'skykatana_pipe'
mkp_local.write(pipe_dir, overwrite=True)
moc_local = SkyMaskPipe.moc_from_stage(mkp_local.mask, order_forced=min(12, mkp_local.order_out), progress=False)
moc_path = OUT_DIR / 'dp2_local_tap_mask_moc.fits'
moc_local.save(str(moc_path), format='fits', overwrite=True)

print('\n=== Final Local TAP SkyMaskPipe Summary ===')
print(mkp_local)

## 5. Multi-Scale Complete Mask Visualization (`SkyMaskPipe.plot_moc`, `pix_in_zone` + `plot`, `plot_fracmap`)

We inspect the complete local TAP mask across macro footprint, fractional coverage, all-sky Mollweide, and sub-arcminute star-halo + bleed-trail scales:

In [ ]:
fig_ms, (ax_ov, ax_zm) = plot_complete_mask_multiscale(
    mkp_local,
    circles_path=star_info['circles_parquet_path'],
    boxes_path=star_info['boxes_fits_path'],
    zoom_half_width_deg=0.25,
    save_path=OUT_DIR / 'dp2_local_tap_complete_mask.png',
)

fig_fr, ax_fr, wcs_fr = plot_complete_mask_fracmap(
    mkp_local,
    save_path=OUT_DIR / 'dp2_local_tap_fracmap.png',
)

fig_mol, ax_mol, wcs_mol = plot_complete_mask_allsky(
    mkp_local,
    save_path=OUT_DIR / 'dp2_local_tap_allsky.png',
)

## 6. End-to-End Science Application: Query `dp2.Object` via TAP, Apply `mkp_local.apply()`, and De-redden Galaxy Colors

Finally, we query extragalactic sources (`refExtendedness = 1`) from `dp2.Object` via TAP, apply our local `SkyMaskPipe` mask (`mkp_local.apply(stage='mask', ...)`), generate matching uniform randoms (`mkp_local.makerans`), and apply **DP2 Tutorial 311.1**'s dust extinction correction to the surviving galaxies' `g, r, i` `cModelMag` photometry:

In [ ]:
adql_galaxies = (
    f"SELECT TOP 25000 objectId, coord_ra, coord_dec, ebv, g_cModelMag, r_cModelMag, i_cModelMag "
    f"FROM {schema_info.object_table} "
    f"WHERE CONTAINS(POINT('ICRS', coord_ra, coord_dec), "
    f"CIRCLE('ICRS', {local_cfg.region.ra_deg:.6f}, {local_cfg.region.dec_deg:.6f}, {local_cfg.region.radius_deg:.6f})) = 1 "
    f"AND refExtendedness = 1"
)
gal_df = tap_client.search(adql_galaxies, maxrec=25_000).to_table().to_pandas()
clean_gal_df = mkp_local.apply(stage='mask', cat=gal_df, columns=('coord_ra', 'coord_dec')).copy()

# De-redden surviving galaxies using Tutorial 311.1 Rubin R_b coefficients
ebv_g = clean_gal_df['ebv'].to_numpy(dtype=float)
for b in ('g', 'r', 'i'):
    clean_gal_df[f'{b}_cModelMag_dered'] = deredden_magnitude(
        clean_gal_df[f'{b}_cModelMag'].to_numpy(dtype=float), ebv_g, band=b, rx_map=rx
    )

# Generate 25,000 uniform randoms strictly inside the surviving mask
rans_df = mkp_local.makerans(stage='mask', nr=25_000, rng=np.random.RandomState(2026))
rans_df.to_parquet(OUT_DIR / 'dp2_local_tap_randoms.parquet', index=False)
clean_gal_df.to_parquet(OUT_DIR / 'dp2_local_tap_clean_galaxies.parquet', index=False)

print(f'Queried TAP galaxies : {len(gal_df):,} -> Clean unmasked galaxies: {len(clean_gal_df):,}')
print(f'Max E(B-V) before cut: {gal_df["ebv"].max():.4f} mag -> Max E(B-V) after mask: {clean_gal_df["ebv"].max():.4f} mag (limit={local_cfg.dust.max_ebv})')
print(f'Median (g - r) shift : {np.nanmedian((clean_gal_df["g_cModelMag"] - clean_gal_df["r_cModelMag"]) - (clean_gal_df["g_cModelMag_dered"] - clean_gal_df["r_cModelMag_dered"])):.4f} mag')
display(clean_gal_df[['objectId', 'coord_ra', 'coord_dec', 'ebv', 'r_cModelMag', 'r_cModelMag_dered']].head())